In [3]:
import sys
from pathlib import Path

# Add project root (APIS/) to sys.path so backend/ and data_generation/ are importable
project_root = Path.cwd().parent  # notebooks/ -> APIS/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# APIS Model Training Pipeline

Trains 4 models:
1. Next Semester GPA (Regression)
2. Final CGPA (Regression)
3. Graduation Class (Classification - 6 classes)
4. Academic Risk (Classification - 3 classes)

In [4]:
import pandas as pd
import numpy as np
import joblib
import json
from datetime import datetime

df = pd.read_csv("../data/synthetic_dataset.csv", dtype={"student_id": str})
print(f"Loaded {len(df)} rows, {df['student_id'].nunique()} students")
from backend.schemas import (
    FEATURE_COLUMNS, TARGET_NEXT_GPA, TARGET_FINAL_CGPA,
    TARGET_GRADUATION_CLASS, TARGET_ACADEMIC_RISK,
    GRADUATION_CLASSES, ACADEMIC_RISK_CLASSES
)

from sklearn.model_selection import train_test_split, GroupKFold, RandomizedSearchCV, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, confusion_matrix, classification_report
)
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import catboost as cb

Loaded 994 rows, 100 students


In [7]:
# Generate / load dataset
from data_generation.generator import generate_dataset
rows = generate_dataset(n_students=15000, programme_durations=[4,5,6], seed=42)
df = pd.DataFrame([r.model_dump() for r in rows])
print(f"Dataset shape: {df.shape}")

Dataset shape: (149956, 25)


In [9]:
# Split BY STUDENT (no leakage)
import numpy as np

student_ids = np.array(df['student_id'].unique(), dtype=object)
train_ids, temp_ids = train_test_split(student_ids, test_size=0.30, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=42)

train_df = df[df['student_id'].isin(train_ids)]
val_df = df[df['student_id'].isin(val_ids)]
test_df = df[df['student_id'].isin(test_ids)]

print(f"Train: {len(train_ids)} students, {len(train_df)} rows")
print(f"Val: {len(val_ids)} students, {len(val_df)} rows")
print(f"Test: {len(test_ids)} students, {len(test_df)} rows")

Train: 10500 students, 105018 rows
Val: 2250 students, 22464 rows
Test: 2250 students, 22474 rows


In [21]:
# Prepare X/y for each target
def prepare_xy(dataframe, target_col):
    y = dataframe[target_col].dropna()
    X = dataframe.loc[y.index, FEATURE_COLUMNS]
    return X, y

X1_train, y1_train = prepare_xy(train_df, TARGET_NEXT_GPA)
X1_val, y1_val = prepare_xy(val_df, TARGET_NEXT_GPA)
X1_test, y1_test = prepare_xy(test_df, TARGET_NEXT_GPA)

X2_train, y2_train = prepare_xy(train_df, TARGET_FINAL_CGPA)
X2_val, y2_val = prepare_xy(val_df, TARGET_FINAL_CGPA)
X2_test, y2_test = prepare_xy(test_df, TARGET_FINAL_CGPA)

X3_train, y3_train = prepare_xy(train_df, TARGET_GRADUATION_CLASS)
X3_val, y3_val = prepare_xy(val_df, TARGET_GRADUATION_CLASS)
X3_test, y3_test = prepare_xy(test_df, TARGET_GRADUATION_CLASS)

X4_train, y4_train = prepare_xy(train_df, TARGET_ACADEMIC_RISK)
X4_val, y4_val = prepare_xy(val_df, TARGET_ACADEMIC_RISK)
X4_test, y4_test = prepare_xy(test_df, TARGET_ACADEMIC_RISK)

# Encode classification targets
# Fit on classes actually present in the data, not the full canonical GRADUATION_CLASSES list —
# "Fail" never occurs in this synthetic dataset, so including it would leave a gap in the
# 0-indexed label range that XGBoost requires. classify_cgpa() in grading_rules.py remains
# 6-class-capable and is the deterministic source of truth for any real edge-case student;
# only this ML classifier's label set is scoped to observed classes.
present_classes = sorted(df['graduation_class'].unique())
le_class = LabelEncoder()
le_class.fit(present_classes)
y3_train_enc = le_class.transform(y3_train)
y3_val_enc = le_class.transform(y3_val)
y3_test_enc = le_class.transform(y3_test)

le_risk = LabelEncoder()
le_risk.fit(['Low', 'Medium', 'High'])
y4_train_enc = le_risk.transform(y4_train)
y4_val_enc = le_risk.transform(y4_val)
y4_test_enc = le_risk.transform(y4_test)

In [11]:
# Training function with CV
def train_with_cv(model_class, param_dist, X, y, cv_groups, task_type='regression', n_iter=50):
    cv = GroupKFold(n_splits=3)
    scoring = 'neg_mean_absolute_error' if task_type == 'regression' else 'f1_macro'
    
    search = RandomizedSearchCV(
        model_class(), param_dist, n_iter=n_iter,
        cv=cv, scoring=scoring, n_jobs=-1, random_state=42, verbose=1
    )
    search.fit(X, y, groups=cv_groups)
    return search.best_estimator_, search.best_params_, search.best_score_

# Model definitions
reg_models = {
    'LinearRegression': (LinearRegression, {}),
    'Ridge': (Ridge, {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}),
    'RandomForest': (RandomForestRegressor, {'n_estimators': [200, 500], 'max_depth': [5, 10, None], 'min_samples_split': [2, 5]}),
    'GradientBoosting': (GradientBoostingRegressor, {'n_estimators': [200, 500], 'learning_rate': [0.01, 0.05, 0.1], 'max_depth': [3, 5]}),
    'XGBoost': (xgb.XGBRegressor, {'n_estimators': [200, 500], 'learning_rate': [0.01, 0.05, 0.1], 'max_depth': [3, 5, 7], 'subsample': [0.8, 1.0]}),
    'CatBoost': (cb.CatBoostRegressor, {'iterations': [200, 500], 'learning_rate': [0.01, 0.05, 0.1], 'depth': [4, 6, 8], 'verbose': [False]}),
}

clf_models = {
    'LogisticRegression': (LogisticRegression, {'C': [0.01, 0.1, 1.0, 10.0], 'max_iter': [1000], 'class_weight': ['balanced']}),
    'RandomForest': (RandomForestClassifier, {'n_estimators': [200, 500], 'max_depth': [5, 10, None], 'min_samples_split': [2, 5], 'class_weight': ['balanced']}),
    'XGBoost': (xgb.XGBClassifier, {'n_estimators': [200, 500], 'learning_rate': [0.01, 0.05, 0.1], 'max_depth': [3, 5, 7]}),
    'CatBoost': (cb.CatBoostClassifier, {'iterations': [200, 500], 'learning_rate': [0.01, 0.05, 0.1], 'depth': [4, 6, 8], 'verbose': [False]}),
}

In [ ]:
# Train all models
def train_task(name, models_dict, X_train, y_train, X_val, y_val, cv_groups, task_type, label_encoder=None):
    print(f"\n=== {name} ===")
    best_model = None
    best_val_score = -np.inf if task_type == 'classification' else np.inf
    best_artifact = None
    
    for model_name, (model_class, param_dist) in models_dict.items():
        print(f"  Training {model_name}...")
        try:
            model, best_params, cv_score = train_with_cv(
                model_class, param_dist, X_train, y_train, cv_groups, task_type
            )
            
            # Evaluate on validation
            if task_type == 'regression':
                val_pred = model.predict(X_val)
                val_mae = mean_absolute_error(y_val, val_pred)
                val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
                val_r2 = r2_score(y_val, val_pred)
                print(f"    CV: {-cv_score:.4f}, Val MAE: {val_mae:.4f}, RMSE: {val_rmse:.4f}, R²: {val_r2:.4f}")
                score = val_mae
                better = score < best_val_score
            else:
                val_pred = model.predict(X_val)
                val_acc = accuracy_score(y_val, val_pred)
                val_f1 = f1_score(y_val, val_pred, average='macro')
                print(f"    CV: {cv_score:.4f}, Val Acc: {val_acc:.4f}, Macro F1: {val_f1:.4f}")
                score = val_f1
                better = score > best_val_score
            
            if better:
                best_val_score = score
                best_model = model
                
                # Feature importance
                if hasattr(model, 'feature_importances_'):
                    fi = list(zip(FEATURE_COLUMNS, model.feature_importances_))
                elif hasattr(model, 'coef_'):
                    fi = list(zip(FEATURE_COLUMNS, np.abs(model.coef_).flatten() if model.coef_.ndim > 1 else np.abs(model.coef_)))
                else:
                    fi = []
                fi = [{'feature': f, 'importance': float(i)} for f, i in fi]
                fi.sort(key=lambda x: x['importance'], reverse=True)
                
                best_artifact = {
                    'model': model,
                    'feature_columns': FEATURE_COLUMNS,
                    'metrics': {'cv_score': float(cv_score), 'val_score': float(score)},
                    'feature_importance': fi,
                }
                if label_encoder is not None:
                    best_artifact['label_encoder'] = label_encoder
        except Exception as e:
            print(f"    Failed: {e}"); import traceback; traceback.print_exc()
    
    return best_model, best_artifact


In [14]:
# Train first tasks
print("Training Next Semester GPA (Regression)")
best_next_gpa, artifact_next_gpa = train_task(
    'Next GPA', reg_models, X1_train, y1_train, X1_val, y1_val, train_df.loc[X1_train.index, 'student_id'], 'regression'
)

Training Next Semester GPA (Regression)

=== Next GPA ===
  Training LinearRegression...
Fitting 3 folds for each of 1 candidates, totalling 3 fits


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 1 is smaller than n_iter=50. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    CV: 0.3220, Val MAE: 0.3233, RMSE: 0.4457, R²: 0.7202
  Training Ridge...
Fitting 3 folds for each of 5 candidates, totalling 15 fits


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 5 is smaller than n_iter=50. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    CV: 0.3220, Val MAE: 0.3233, RMSE: 0.4457, R²: 0.7202
  Training RandomForest...
Fitting 3 folds for each of 12 candidates, totalling 36 fits


...

    CV: 0.3155, Val MAE: 0.3153, RMSE: 0.4363, R²: 0.7319
  Training XGBoost...
Fitting 3 folds for each of 36 candidates, totalling 108 fits


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 36 is smaller than n_iter=50. Running 36 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    CV: 0.3151, Val MAE: 0.3151, RMSE: 0.4361, R²: 0.7322
  Training CatBoost...
Fitting 3 folds for each of 18 candidates, totalling 54 fits


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 18 is smaller than n_iter=50. Running 18 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    CV: 0.3144, Val MAE: 0.3149, RMSE: 0.4357, R²: 0.7326


In [15]:
# train second task
print("\nTraining Final CGPA (Regression)")
best_final_cgpa, artifact_final_cgpa = train_task(
    'Final CGPA', reg_models, X2_train, y2_train, X2_val, y2_val, train_df.loc[X2_train.index, 'student_id'], 'regression'
)


Training Final CGPA (Regression)

=== Final CGPA ===
  Training LinearRegression...
Fitting 3 folds for each of 1 candidates, totalling 3 fits


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 1 is smaller than n_iter=50. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    CV: 0.1281, Val MAE: 0.1280, RMSE: 0.1991, R²: 0.9275
  Training Ridge...
Fitting 3 folds for each of 5 candidates, totalling 15 fits


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 5 is smaller than n_iter=50. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    CV: 0.1281, Val MAE: 0.1280, RMSE: 0.1991, R²: 0.9275
  Training RandomForest...
Fitting 3 folds for each of 12 candidates, totalling 36 fits


...

    CV: 0.1190, Val MAE: 0.1180, RMSE: 0.1870, R²: 0.9361
  Training XGBoost...
Fitting 3 folds for each of 36 candidates, totalling 108 fits


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 36 is smaller than n_iter=50. Running 36 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    CV: 0.1187, Val MAE: 0.1179, RMSE: 0.1873, R²: 0.9359
  Training CatBoost...
Fitting 3 folds for each of 18 candidates, totalling 54 fits


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 18 is smaller than n_iter=50. Running 18 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    CV: 0.1186, Val MAE: 0.1183, RMSE: 0.1872, R²: 0.9360


In [22]:
#train third task
print("\nTraining Graduation Class (Classification)")
best_grad_class, artifact_grad_class = train_task(
    'Grad Class', clf_models, X3_train, y3_train_enc, X3_val, y3_val_enc, train_df.loc[X3_train.index, 'student_id'], 'classification', le_class
)


Training Graduation Class (Classification)

=== Grad Class ===
  Training LogisticRegression...
Fitting 3 folds for each of 4 candidates, totalling 12 fits


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=50. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_

    CV: 0.6212, Val Acc: 0.7610, Macro F1: 0.6182
  Training RandomForest...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
    CV: 0.7760, Val Acc: 0.8863, Macro F1: 0.8078
  Training XGBoost...
Fitting 3 folds for each of 18 candidates, totalling 54 fits


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 18 is smaller than n_iter=50. Running 18 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    CV: 0.7813, Val Acc: 0.8922, Macro F1: 0.7876
  Training CatBoost...
Fitting 3 folds for each of 18 candidates, totalling 54 fits


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 18 is smaller than n_iter=50. Running 18 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    CV: 0.7804, Val Acc: 0.8938, Macro F1: 0.8329


In [31]:
#train final task
print("\nTraining Academic Risk (Classification)")
best_acad_risk, artifact_acad_risk = train_task(
    'Acad Risk', clf_models, X4_train, y4_train_enc, X4_val, y4_val_enc, train_df.loc[X4_train.index, 'student_id'], 'classification', le_risk
)


Training Academic Risk (Classification)

=== Acad Risk ===
  Training LogisticRegression...
Fitting 3 folds for each of 4 candidates, totalling 12 fits


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=50. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_

    CV: 0.8871, Val Acc: 0.9378, Macro F1: 0.8794
  Training RandomForest...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
    CV: 0.9998, Val Acc: 1.0000, Macro F1: 1.0000
  Training XGBoost...
Fitting 3 folds for each of 18 candidates, totalling 54 fits


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 18 is smaller than n_iter=50. Running 18 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    CV: 0.9895, Val Acc: 0.9973, Macro F1: 0.9869
  Training CatBoost...
Fitting 3 folds for each of 18 candidates, totalling 54 fits


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 18 is smaller than n_iter=50. Running 18 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    CV: 0.9944, Val Acc: 0.9976, Macro F1: 0.9879


In [32]:
# Final evaluation on test set
def evaluate_final(model, X_test, y_test, task_type, label_encoder=None, target_name=''):
    pred = model.predict(X_test)
    if task_type == 'regression':
        mae = mean_absolute_error(y_test, pred)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        r2 = r2_score(y_test, pred)
        print(f"{target_name} Test: MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}")
        return {'mae': mae, 'rmse': rmse, 'r2': r2}
    else:
        if label_encoder:
            y_test_labels = label_encoder.inverse_transform(y_test)
            pred_labels = label_encoder.inverse_transform(pred)
        else:
            y_test_labels = y_test
            pred_labels = pred
        acc = accuracy_score(y_test_labels, pred_labels)
        f1 = f1_score(y_test_labels, pred_labels, average='macro')
        print(f"{target_name} Test: Acc={acc:.4f}, Macro F1={f1:.4f}")
        print(classification_report(y_test_labels, pred_labels))
        # Confusion matrix check
        cm = confusion_matrix(y_test_labels, pred_labels, normalize='true')
        classes = label_encoder.classes_ if label_encoder else np.unique(y_test_labels)
        for i, cls in enumerate(classes):
            for j, cls2 in enumerate(classes):
                if i != j and cm[i, j] > 0.20:
                    print(f"  WARNING: {cls} -> {cls2}: {cm[i, j]:.1%} > 20%")
        return {'accuracy': acc, 'macro_f1': f1}

print("\n=== FINAL TEST EVALUATION ===")
metrics_next = evaluate_final(best_next_gpa, X1_test, y1_test, 'regression', target_name='Next GPA')
metrics_final = evaluate_final(best_final_cgpa, X2_test, y2_test, 'regression', target_name='Final CGPA')
metrics_class = evaluate_final(best_grad_class, X3_test, y3_test_enc, 'classification', le_class, 'Graduation Class')
metrics_risk = evaluate_final(best_acad_risk, X4_test, y4_test_enc, 'classification', le_risk, 'Academic Risk')


=== FINAL TEST EVALUATION ===
Next GPA Test: MAE=0.3125, RMSE=0.4349, R²=0.7392
Final CGPA Test: MAE=0.1160, RMSE=0.1845, R²=0.9390


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\preprocessing\_label.py:161: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Graduation Class Test: Acc=0.8852, Macro F1=0.7961
                    precision    recall  f1-score   support

       First Class       0.85      0.71      0.78      1492
              Pass       0.91      0.39      0.55        82
Second Class Lower       0.88      0.88      0.88      7958
Second Class Upper       0.89      0.92      0.91     10598
       Third Class       0.89      0.85      0.87      2344

          accuracy                           0.89     22474
         macro avg       0.88      0.75      0.80     22474
      weighted avg       0.88      0.89      0.88     22474

Academic Risk Test: Acc=1.0000, Macro F1=1.0000
              precision    recall  f1-score   support

        High       1.00      1.00      1.00       903
         Low       1.00      1.00      1.00     16724
      Medium       1.00      1.00      1.00      4847

    accuracy                           1.00     22474
   macro avg       1.00      1.00      1.00     22474
weighted avg       1.00      1.0

In [6]:
# Save models
import os
os.makedirs('models', exist_ok=True)

joblib.dump(artifact_next_gpa, 'models/next_gpa.pkl')
joblib.dump(artifact_final_cgpa, 'models/final_cgpa.pkl')
joblib.dump(artifact_grad_class, 'models/graduation_class.pkl')
joblib.dump(artifact_acad_risk, 'models/academic_risk.pkl')

print("Models saved to models/")

NameError: name 'artifact_next_gpa' is not defined

In [34]:
# Log training run
log_entry = {
    'timestamp': datetime.now().isoformat(),
    'task': 'next_semester_gpa',
    'model_type': type(best_next_gpa).__name__,
    'hyperparameters': best_next_gpa.get_params() if hasattr(best_next_gpa, 'get_params') else {},
    'cv_mae': -artifact_next_gpa['metrics']['cv_score'],
    'val_mae': artifact_next_gpa['metrics']['val_score'],
    'test_mae': metrics_next['mae'],
    'test_rmse': metrics_next['rmse'],
    'test_r2': metrics_next['r2'],
    'selected': True
}

with open('models/training_log.jsonl', 'a') as f:
    f.write(json.dumps(log_entry) + '\n')

# Repeat for other 3 models...
print("Training log appended to models/training_log.jsonl")

Training log appended to models/training_log.jsonl


In [35]:
# Add the generate_dataset import at the top of the notebook
from data_generation.generator import generate_dataset